In [45]:
!pip install spacy
!pip install sentence-transformers

!python -m spacy download en_core_web_sm
!python -m spacy download es_core_news_sm


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 35.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


In [46]:
import os
import re
import contractions
import json
import spacy
from sentence_transformers import SentenceTransformer, util
import numpy as np
from scipy.optimize import linear_sum_assignment

In [47]:
def detectar_idioma(archivo_ass):
    with open(archivo_ass, "r", encoding="utf-8") as f:
        for linea in f:
            match = re.search(r"^Title:\s*(.*)", linea, re.IGNORECASE)
            if match:
                return match.group(1).strip()
    return "Idioma desconocido"

In [48]:
idioma_map = {
    "español": "es",
    "english": "en"
}

def obtener_codigo_idioma(texto):
    # Convertir a minúsculas para evitar problemas de mayúsculas
    texto = texto.lower()
    
    # Buscar idioma en el mapeo
    for clave, codigo in idioma_map.items():
        if clave in texto:
            return codigo
    
    return "desconocido"  # Si no se reconoce el idioma

In [49]:
def extract_subtitles_with_timestamps(ass_file, output_txt):
    with open(ass_file, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    subtitles = []
    in_events = False

    for line in lines:
        # Detecta el inicio de la sección [Events]
        if line.strip().lower() == "[events]":
            in_events = True
            continue

        if line.strip().lower() == "title:":
            in_events = True
            continue

        if in_events:
            # Extrae solo las líneas de subtítulos (que empiezan con "Dialogue:")
            if line.startswith("Dialogue:"):
                # Divide la línea en columnas usando la coma como separador
                parts = line.split(",", 9)  # Separa en máximo 10 partes
                if len(parts) > 9:
                    author = parts[4].strip()    # Tiempo de fin
                    subtitle_text = parts[9].strip()  # El texto del subtítulo
                    # Formatea la salida incluyendo las marcas de tiempo
                    subtitles.append(f"{author},{subtitle_text}")

    # Guarda los subtítulos en un archivo .txt
    with open(output_txt, 'w', encoding='utf-8') as out_file:
        out_file.write("\n".join(subtitles))


In [50]:
def process_folder(folder_path, output_folder):
    # Crea la carpeta de salida si no existe
    os.makedirs(output_folder, exist_ok=True)

    # Itera sobre todos los archivos .ass en el folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".ass"):
            ass_path = os.path.join(folder_path, file_name)
            idioma = obtener_codigo_idioma(detectar_idioma(ass_path))
            file_name = idioma + "." + file_name
            txt_path = os.path.join(output_folder, file_name.replace(".ass", ".txt"))

            print(f"Procesando: {file_name} -> {txt_path}")
            extract_subtitles_with_timestamps(ass_path, txt_path)

In [51]:
def limpiar_texto(texto):
    """Elimina signos de puntuación y caracteres no alfabéticos."""
    # texto = re.sub(r'[^a-zA-Z\s¿?¡!]', '', texto)
    texto = re.sub(r'\{.*?\}', '', texto)
    return texto.lower().strip()

In [52]:
def expand_contractions(text):
    return contractions.fix(text)

In [53]:
def renombrar_archivo(nombre):
    match = re.search(r"\[(?:.*?)\]\s*(.*?)\s*-\s*(\d+).*?(\.\d+\..*?)$", nombre)
    if match:
        nuevo_nombre = f"{match.group(1).replace(' ', '')}{match.group(2)}{match.group(3)}"
        return nuevo_nombre
    return nombre

In [54]:
def group_subtitles_by_author_to_files(input_file, output_dir):
    """
    Lee un archivo con formato "autor,texto", agrupa las líneas consecutivas del mismo autor
    y guarda, para cada autor, un archivo independiente en 'output_dir' con los grupos concatenados.
    """
    # Diccionario para almacenar grupos por autor.
    # Cada clave será un autor y su valor una lista de grupos (secuencias consecutivas).
    grouped = {}

    current_author = None
    current_text = ""

    # Lee el archivo de entrada
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Recorre línea a línea
    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Separa la línea en dos partes: autor y texto (máximo 1 división)
        parts = line.split(",", 1)
        if len(parts) != 2:
            continue  # Si la línea no cumple el formato, la omite

        author, text = parts[0].strip(), parts[1].strip()

        # Si es la primera línea o si el autor es el mismo que el anterior, se concatena el texto
        if current_author is None or author == current_author:
            if current_author is None:
                current_author = author
            current_text += (" " if current_text else "") + text
        else:
            # Cuando el autor cambia, se guarda el grupo anterior en el diccionario
            if current_author not in grouped:
                grouped[current_author] = []
            grouped[current_author].append(current_text)

            # Se reinicia el grupo para el nuevo autor
            current_author = author
            current_text = text

    # Guarda el último grupo
    if current_author is not None:
        if current_author not in grouped:
            grouped[current_author] = []
        grouped[current_author].append(current_text)

    # Asegúrate de que el directorio de salida exista
    os.makedirs(output_dir, exist_ok=True)

    # Escribe un archivo para cada autor con los grupos concatenados (separados por una línea en blanco)
    for author, groups in grouped.items():
        # Genera un nombre de archivo simple (puedes mejorar la sanitización si es necesario)
        filename = os.path.join(output_dir, f"{author.replace("/", "_") if author else 'sin_autor'}.txt")
        with open(filename, 'a', encoding='utf-8') as out_file:
            out_file.write("\n\n".join(groups))

In [55]:
# Cargar modelos de spaCy para segmentar oraciones
nlp_en = spacy.load("en_core_web_sm")
nlp_es = spacy.load("es_core_news_sm")

def segment_text(text, nlp):
    """
    Segmenta el texto en oraciones usando spaCy.
    """
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    return sentences

In [56]:
def align_sentences_global(english_text, spanish_text):
    """
    Segmenta ambos textos en oraciones y alinea globalmente cada oración en inglés
    con la oración en español que, en conjunto, maximice la similitud total.
    
    Retorna una lista de tuplas (oración_en, oración_es, similitud).
    """
    # Segmentar los textos
    sentences_en = segment_text(english_text, nlp_en)
    sentences_es = segment_text(spanish_text, nlp_es)
    
    # Cargar el modelo multilingüe para obtener embeddings
    model = SentenceTransformer('distiluse-base-multilingual-cased-v2')
    
    # Generar embeddings para ambas listas de oraciones
    embeddings_en = model.encode(sentences_en, convert_to_tensor=True)
    embeddings_es = model.encode(sentences_es, convert_to_tensor=True)
    
    # Calcular la matriz de similitud coseno
    cosine_scores = util.cos_sim(embeddings_en, embeddings_es).cpu().numpy()  # forma: [n_en, n_es]
    
    # Convertir la similitud en un costo (el algoritmo húngaro minimiza el costo)
    cost_matrix = -cosine_scores
    
    # Usar el algoritmo húngaro para obtener el emparejamiento global óptimo
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    aligned_pairs = []
    for i, j in zip(row_ind, col_ind):
        aligned_pairs.append((sentences_en[i], sentences_es[j], float(cosine_scores[i][j])))
    
    return aligned_pairs

In [57]:
def save_aligned_to_json(aligned, output_file, threshold=0.9):
    """
    Guarda en un archivo JSON cada par alineado con score >= threshold.
    Cada registro tendrá los campos: "score", "en" y "es".
    """
    results = []
    contador = 0
    for en, es, score in aligned:
        if score >= threshold:
            results.append({
                "score": score,
                "en": en.replace("\n", " "),
                "es": es.replace("\n", " ")
            })
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            contador += 1
    return contador

In [58]:
output_folder = "animes_1"
process_folder("animes", output_folder)

Procesando: es.JUJUTSU KAISEN Season 2 Episode 30.Español (LA).ass -> animes_1/es.JUJUTSU KAISEN Season 2 Episode 30.Español (LA).txt
Procesando: es.JUJUTSU KAISEN Season 2 Episode 38.Español (LA).ass -> animes_1/es.JUJUTSU KAISEN Season 2 Episode 38.Español (LA).txt
Procesando: es.JUJUTSU KAISEN Episodio 10 esLA.ass -> animes_1/es.JUJUTSU KAISEN Episodio 10 esLA.txt
Procesando: es.JUJUTSU KAISEN Episodio 23 esLA.ass -> animes_1/es.JUJUTSU KAISEN Episodio 23 esLA.txt
Procesando: en.JUJUTSU KAISEN Season 2 Episode 35.English.ass -> animes_1/en.JUJUTSU KAISEN Season 2 Episode 35.English.txt
Procesando: es.[Erai-raws] Jujutsu Kaisen - 06 [1080p][Multiple Subtitle].9.spa.ass -> animes_1/es.[Erai-raws] Jujutsu Kaisen - 06 [1080p][Multiple Subtitle].9.spa.txt
Procesando: es.JUJUTSU KAISEN Episodio 14 esLA.ass -> animes_1/es.JUJUTSU KAISEN Episodio 14 esLA.txt
Procesando: es.JUJUTSU KAISEN Season 2 Episode 34.Español (LA).ass -> animes_1/es.JUJUTSU KAISEN Season 2 Episode 34.Español (LA).txt


In [59]:
# Realizar limpieza de los archivos ya unificados

os.makedirs(output_folder, exist_ok=True)

output_folder_final_2 = "animes_2"
os.makedirs(output_folder_final_2, exist_ok=True)

for file_name in os.listdir(output_folder):
    with open(output_folder + "/" + file_name, 'r', encoding='utf-8') as file:
        content = file.read().replace("\\N", " ")
        content_expandido = expand_contractions(content)
        texto_limpio = limpiar_texto(content_expandido)
        file.close()

        with open(output_folder_final_2 + "/" + renombrar_archivo(file_name), 'w', encoding='utf-8') as file:
            file.write(texto_limpio)

In [60]:

output_folder_final_3 = "animes_3"
os.makedirs(output_folder_final_3, exist_ok=True)

output_dir_es = output_folder_final_3 + "/" + "es"
output_dir_en = output_folder_final_3 + "/" + "en"
os.makedirs(output_dir_es, exist_ok=True)
os.makedirs(output_dir_en, exist_ok=True)

target_folder = "./" + output_folder_final_2

for file_name in os.listdir(target_folder):
    input_file = target_folder + "/" + file_name
    if file_name.startswith("es"):
        group_subtitles_by_author_to_files(input_file, output_dir_es)
    elif file_name.startswith("en"):
        group_subtitles_by_author_to_files(input_file, output_dir_en)


In [61]:
output_folder_final = "animes_4"
os.makedirs(output_folder_final, exist_ok=True)

# Alinear oraciones
archivos_es = set(os.listdir(output_dir_es))
archivos_en = set(os.listdir(output_dir_en))

archivos_comunes = archivos_es.intersection(archivos_en)
contador = 0
for file_name in archivos_comunes:
    file_name_es = "./"+ os.path.join(output_dir_es, file_name)
    file_name_en = "./" + os.path.join(output_dir_en, file_name)
    file_name_es_content = open(file_name_es, "r")
    file_name_en_content = open(file_name_en, "r")
    aligned = align_sentences_global(file_name_en_content.read(), file_name_es_content.read())
    file_name_es_content.close()
    file_name_en_content.close()
    contador += save_aligned_to_json(aligned, f"./{output_folder_final}/{file_name}.json", threshold=0.87)

print("Frases utiles: ", contador)

Frases utiles:  669
